<a href="https://colab.research.google.com/github/PratikshaShind/OASIS-INFOBYTE/blob/main/Task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import io
import numpy as np
import pandas as pd
from google.colab import files

print("💡 STEP 1: Upload your messy CSV file...")
print(
    "👉 Note: If you do not have a file, click 'Cancel'. The script will automatically generate a messy synthetic dataset for you!"
)


uploaded = files.upload()

if len(uploaded) > 0:
    # Load user-uploaded file
    filename = list(uploaded.keys())[0]
    df = pd.read_csv(io.BytesIO(uploaded[filename]))
    print(f"\n[SUCCESS] '{filename}' loaded successfully!")
else:
    # Fallback to a synthetic messy dataset if no file is uploaded
    print(
        "\n[INFO] No file uploaded. Generating a synthetically messy dataset for demonstration..."
    )
    raw_data = {
        "User_ID": [
            "USR_1001",
            "1002",
            "USR_1003",
            "USR_1001",
            "USR_1004",
            "1005",
            "USR_1006",
            "USR_1007",
        ],
        "Gender": [
            "Male",
            "female",
            "M",
            "Male",
            "F",
            "Female",
            "male",
            "Female",
        ],
        "Salary": [
            "$85,000",
            "$92,000",
            None,
            "$85,000",
            "$500,000",
            "$78,500",
            "-$10,000",
            "$64,000",
        ],
        "Join_Date": [
            "2023/01/15",
            "15-02-2023",
            "2023.03.20",
            "2023/01/15",
            None,
            "2023-05-12",
            "01/06/2023",
            "2023-07-19",
        ],
    }
    df = pd.DataFrame(raw_data)

print("\n--- Initial (Messy) Dataset Preview ---")
display(df.head(10))
print("\n" + "=" * 60 + "\n")



def generate_quality_report(dataframe, stage_name):
    print(f"=== 📊 Data Quality Report: {stage_name} ===")
    print(f"Total Rows: {len(dataframe)}")
    print(f"Duplicate Rows: {dataframe.duplicated().sum()}")
    print("\n--- Column-wise Overview ---")

    report_df = pd.DataFrame(
        {
            "Data Type": dataframe.dtypes,
            "Null Count": dataframe.isnull().sum(),
            "Null %": (dataframe.isnull().sum() / len(dataframe)) * 100,
        }
    )
    display(report_df)
    print("\n" + "-" * 40 + "\n")


# Generate and save pre-cleaning metrics
generate_quality_report(df, "Before Cleaning")

before_metrics = {
    "row_count": len(df),
    "duplicate_count": df.duplicated().sum(),
    "null_counts": df.isnull().sum().to_dict(),
    "dtypes": df.dtypes.to_dict(),
}


initial_rows = len(df)
df = df.drop_duplicates(keep="first").reset_index(drop=True)
removed_duplicates = initial_rows - len(df)
print(f"🧼 [DELETE] Duplicate resolution: {removed_duplicates} rows removed.\n")


if "User_ID" in df.columns:
    df["User_ID"] = df["User_ID"].astype(str).str.replace("USR_", "")
    df["User_ID"] = "USR_" + df["User_ID"]

if "Gender" in df.columns:
    gender_map = {
        "Male": "Male",
        "male": "Male",
        "M": "Male",
        "Female": "Female",
        "female": "Female",
        "F": "Female",
    }
    df["Gender"] = df["Gender"].map(gender_map)

if "Salary" in df.columns:
    df["Salary"] = df["Salary"].astype(str).str.replace("$", "", regex=False)
    df["Salary"] = df["Salary"].str.replace(",", "", regex=False)
    df["Salary"] = pd.to_numeric(df["Salary"], errors="coerce")
    df["Salary"] = df["Salary"].abs()  # Fix negative salary entry anomalies

print("🔄 [STANDARDIZED] Inconsistent formats normalized successfully.\n")


if "Salary" in df.columns:
    # Use Median Imputation to prevent distortions from numerical outliers
    salary_median = df["Salary"].median()
    df["Salary"] = df["Salary"].fillna(salary_median)

if "Join_Date" in df.columns:
    # Parse mixed string expressions into datetime objects
    df["Join_Date"] = pd.to_datetime(
        df["Join_Date"], format="mixed", errors="coerce"
    )
    # Apply Forward Fill assuming chronological system logging rules
    df["Join_Date"] = df["Join_Date"].ffill()

print("🩹 [IMPUTED] Missing value gaps resolved successfully.\n")


if "Salary" in df.columns:
    Q1 = df["Salary"].quantile(0.25)
    Q3 = df["Salary"].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Cap outliers to statistical limits instead of discarding valid row data
    df["Salary"] = np.where(df["Salary"] > upper_bound, upper_bound, df["Salary"])
    df["Salary"] = np.where(df["Salary"] < lower_bound, lower_bound, df["Salary"])
    print(
        "📈 [OUTLIERS] IQR evaluation complete: Values capped within boundaries.\n"
    )

if "User_ID" in df.columns:
    df["User_ID"] = df["User_ID"].astype("string")
if "Gender" in df.columns:
    df["Gender"] = df["Gender"].astype("category")
if "Salary" in df.columns:
    df["Salary"] = df["Salary"].astype("float64")
if "Join_Date" in df.columns:
    df["Join_Date"] = pd.to_datetime(df["Join_Date"])

print("🎯 [DTYPES] Dynamic casting matching targeted feature structures.\n")
print("=" * 60 + "\n")


generate_quality_report(df, "After Cleaning")

after_metrics = {
    "row_count": len(df),
    "duplicate_count": df.duplicated().sum(),
    "null_counts": df.isnull().sum().to_dict(),
    "dtypes": df.dtypes.to_dict(),
}

summary_data = []
for col in df.columns:
    b_dtype = str(before_metrics["dtypes"].get(col, "N/A"))
    b_null = before_metrics["null_counts"].get(col, 0)

    summary_data.append(
        {
            "Column": col,
            "Before_Dtype": b_dtype,
            "After_Dtype": str(after_metrics["dtypes"][col]),
            "Before_Nulls": b_null,
            "After_Nulls": after_metrics["null_counts"][col],
        }
    )

summary_df = pd.DataFrame(summary_data)

print("=== 📋 Before vs. After Summary Matrix ===")
print(
    f"Initial Row Volume: {before_metrics['row_count']} | Post-Cleaning Row Volume: {after_metrics['row_count']}"
)
print(
    f"Initial Duplicate Rows: {before_metrics['duplicate_count']} | Post-Cleaning Duplicate Rows: {after_metrics['duplicate_count']}\n"
)
display(summary_df)
print("\n" + "=" * 60 + "\n")

output_filename = "cleaned_dataset_output.csv"
df.to_csv(output_filename, index=False)

print(f"🎉 [COMPLETE] Cleansed dataset exported to '{output_filename}'.")
print("📥 Initializing local machine browser download window...\n")

# Auto-downloads file to your device via Google Colab API
files.download(output_filename)

💡 STEP 1: Upload your messy CSV file...
👉 Note: If you do not have a file, click 'Cancel'. The script will automatically generate a messy synthetic dataset for you!


Saving google_cleaned (2).csv to google_cleaned (2).csv

[SUCCESS] 'google_cleaned (2).csv' loaded successfully!

--- Initial (Messy) Dataset Preview ---


,Unnamed: 0,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Current Ver,Android Ver,Day,month,year
0,0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19000.0,10000,Free,0.0,Everyone,Art & Design,1.0.0,4.0.3 and up,7,1,2018
1,1,Coloring book moana,ART_AND_DESIGN,3.9,967,14000.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,2.0.0,4.0.3 and up,15,1,2018
2,2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8700.0,5000000,Free,0.0,Everyone,Art & Design,1.2.4,4.0.3 and up,1,8,2018
3,3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25000.0,50000000,Free,0.0,Teen,Art & Design,Varies with device,4.2 and up,8,6,2018
4,4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2800.0,100000,Free,0.0,Everyone,Art & Design;Creativity,1.1,4.4 and up,20,6,2018
5,5,Paper flowers instructions,ART_AND_DESIGN,4.4,167,5600.0,50000,Free,0.0,Everyone,Art & Design,1.0,2.3 and up,26,3,2017
6,6,Smoke Effect Photo Maker - Smoke Editor,ART_AND_DESIGN,3.8,178,19000.0,50000,Free,0.0,Everyone,Art & Design,1.1,4.0.3 and up,26,4,2018
7,7,Infinite Painter,ART_AND_DESIGN,4.1,36815,29000.0,1000000,Free,0.0,Everyone,Art & Design,6.1.61.1,4.2 and up,14,6,2018
8,8,Garden Coloring Book,ART_AND_DESIGN,4.4,13791,33000.0,1000000,Free,0.0,Everyone,Art & Design,2.9.2,3.0 and up,20,9,2017
9,9,Kids Paint Free - Drawing Fun,ART_AND_DESIGN,4.7,121,3100.0,10000,Free,0.0,Everyone,Art & Design;Creativity,2.8,4.0.3 and up,3,7,2018




=== 📊 Data Quality Report: Before Cleaning ===
Total Rows: 10840
Duplicate Rows: 0

--- Column-wise Overview ---


,Data Type,Null Count,Null %
Unnamed: 0,int64,0,0.0
App,object,0,0.0
Category,object,0,0.0
Rating,float64,0,0.0
Reviews,int64,0,0.0
Size,float64,0,0.0
Installs,int64,0,0.0
Type,object,0,0.0
Price,float64,0,0.0
Content Rating,object,0,0.0


...


----------------------------------------

=== 📋 Before vs. After Summary Matrix ===
Initial Row Volume: 10840 | Post-Cleaning Row Volume: 10840
Initial Duplicate Rows: 0 | Post-Cleaning Duplicate Rows: 0



,Column,Before_Dtype,After_Dtype,Before_Nulls,After_Nulls
0,Unnamed: 0,int64,int64,0,0
1,App,object,object,0,0
2,Category,object,object,0,0
3,Rating,float64,float64,0,0
4,Reviews,int64,int64,0,0
5,Size,float64,float64,0,0
6,Installs,int64,int64,0,0
7,Type,object,object,0,0
8,Price,float64,float64,0,0
9,Content Rating,object,object,0,0




🎉 [COMPLETE] Cleansed dataset exported to 'cleaned_dataset_output.csv'.
📥 Initializing local machine browser download window...



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>